# 13. LOT 240121 · SPCE

## 냉각 온도의 반복 변화와 같은 SPCE 비교

관측: **2024-11-24 07:31:00 ~ 2024-11-24 18:22:12**

1. TC-OUT2는 계속 상승하는가, 오르내림을 반복하는가?
2. 037·133과 냉각 수준과 폭이 다른가?
3. 가열·냉각 최고점의 시차를 소재 이동 시간으로 볼 수 있는가?

## 읽기 전에


원본 온도·조업이벤트·정비이력·정기수리·부품 CSV에서 독립 실행한다. 다른 노트북이나 가공 CSV를 먼저 실행할 필요가 없다. 센서마다 1분 내 유효 관측이 30개 이상일 때만 평균을 사용하고, 부족한 분과 LOT 사이 공백은 보간하지 않는다. 원시 그림도 누락 시각을 이어 붙이지 않는다.

자료는 교육용 재구성 데이터다. 데이터사전에서 온도 ℃와 출력 % 및 공정 구역 배정은 **가정**이고, CP-Z4의 실제 단위는 **확인되지 않았다**. OP의 0은 실제 제어출력으로 취급한다. CP-Z4M은 0이 아니라 작은 스케일의 중복 정보이므로 이 검토의 초점 센서에서 제외했다. 강종에 따른 출력 수준 이동도 검증된 실제 제철소 설정값을 뜻하지 않는다.

표의 기준은 원시 관측과 1분 평균을 구분해 표시했다. 표의 13:29는 13:29:00~59 구간의 평균이며, 곡선은 그 분의 가운데(13:29:30)에 표시한다. 해당 분이 끝난 뒤 계산할 수 있는 값이다. 아래 본문 표는 이번 원자료 실행에서 계산한 값이며, 원자료가 바뀌면 코드의 재실행 결과를 다시 확인한다. 이 노트북은 고장 정답을 만들거나 모델을 학습하지 않는다.

In [ ]:
LOT = 240121
NOTEBOOK_NAME = '13_LOT_240121'
PLOT_TITLE = '냉각 온도의 반복 변화와 같은 SPCE 비교'
FOCUS = ['TC-OUT1', 'TC-OUT2', 'TC-Z3', 'TC-Z8', 'OP-Z4']
COMPARE = ['TC-OUT2', 'TC-Z8']
ZOOM_START = '2024-11-24 16:30'
ZOOM_END = '2024-11-24 17:05'
extra_tables = []

# 팀 공통 경로·인코딩·한글 폰트 설정입니다.
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
import matplotlib.dates as mdates
from IPython.display import display

# 이 하위 폴더, notebooks 폴더, 저장소 루트에서 모두 실행할 수 있습니다.
# data 또는 data/raw 중 선택된 한 폴더에서 모든 원본 CSV를 읽습니다.
for candidate in [os.path.join('..', '..', 'data'), os.path.join('..', 'data'), 'data']:
    if os.path.isfile(os.path.join(candidate, 'T-CR1-CAL01_온도.csv')):
        DATA_DIR = candidate
        break
    raw_candidate = os.path.join(candidate, 'raw')
    if os.path.isfile(os.path.join(raw_candidate, 'T-CR1-CAL01_온도.csv')):
        DATA_DIR = raw_candidate
        break
else:
    raise FileNotFoundError('저장소의 data 또는 data/raw 폴더에 원본 CSV를 함께 넣어 주세요.')

# 원본이 data/raw에 있어도 그림은 저장소의 figures에 저장합니다.
ROOT = Path(candidate).resolve().parent
FIG_DIR = os.path.join(ROOT, 'figures', '07_lot_anomaly_review_namelesspark', NOTEBOOK_NAME)
os.makedirs(FIG_DIR, exist_ok=True)
TEMP_CSV = os.path.join(DATA_DIR, 'T-CR1-CAL01_온도.csv')
EVENT_CSV = os.path.join(DATA_DIR, 'G-02_조업이벤트.csv')
REPAIR_CSV = os.path.join(DATA_DIR, 'G-01_정기수리캘린더.csv')
PART_CSV = os.path.join(DATA_DIR, 'G-01_부품마스터.csv')
MAINT_CSV = os.path.join(DATA_DIR, 'G-02_정비이력.csv')
PARTCHG_CSV = os.path.join(DATA_DIR, 'G-02_부품교체이력.csv')
ENC = 'utf-8-sig'

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR']:
    if cand in installed:
        rcParams['font.family'] = cand
        break
else:
    print('경고: 한글 폰트를 찾지 못했습니다.')
rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 110
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False

## 원자료에서 LOT와 강종, 유효 1분 관측 만들기

LOT 시작 시각과 CR1 강종변경 시각을 연결한다. 센서별 관측 수를 따로 세어, 결측이 많은 센서의 분을 잘못 유효 처리하지 않는다.

In [ ]:
raw = pd.read_csv(TEMP_CSV, encoding=ENC, parse_dates=['MEAS_DT'])
events = pd.read_csv(EVENT_CSV, encoding=ENC, parse_dates=['EVT_DT'])
maintenance = pd.read_csv(MAINT_CSV, encoding=ENC, parse_dates=['DETC_DT', 'STRT_DT', 'END_DT'])
repair = pd.read_csv(REPAIR_CSV, encoding=ENC, parse_dates=['STRT_DT', 'END_DT'])
parts = pd.read_csv(PART_CSV, encoding=ENC)
part_changes = pd.read_csv(PARTCHG_CSV, encoding=ENC)

sensor_columns = [c for c in raw.columns if c not in ['LOT_NO', 'MEAS_DT', 'CP-Z4M']]
heat_columns = ['TC-Z3', 'TC-Z4', 'TC-Z5', 'TC-Z6']
lot_bounds = raw.groupby('LOT_NO').MEAS_DT.agg(start='min', end='max')
grade_events = events.loc[events.PLANT_CD.eq('CR1') & events.EVT_TYPE.eq('강종변경')]
# 연결이 중복되면 LOT 수가 늘어나므로 일대일 대응을 확인합니다.
lot_bounds = lot_bounds.reset_index()
lot_bounds = lot_bounds.merge(
    grade_events[['EVT_DT', 'AFT_VAL']],
    left_on='start',
    right_on='EVT_DT',
    how='left',
    validate='one_to_one',
)
lot_bounds = lot_bounds.set_index('LOT_NO')
lot_bounds = lot_bounds.rename(columns={'AFT_VAL': 'grade'})

# LOT별 1분 칸을 유지합니다. 관측 30개 미만인 센서 값은 비워 둡니다.
minute_by_lot = {}
count_by_lot = {}
for lot_no, group in raw.groupby('LOT_NO', sort=True):
    series = group.sort_values('MEAS_DT').set_index('MEAS_DT')
    means = series[sensor_columns].resample('1min').mean()
    counts = series[sensor_columns].resample('1min').count()
    means = means.mask(counts.lt(30))
    means['HEAT_MEAN'] = means[heat_columns].mean(axis=1, skipna=False)
    minute_by_lot[lot_no] = means
    count_by_lot[lot_no] = counts

raw_lot = raw.loc[raw.LOT_NO.eq(LOT)].sort_values('MEAS_DT').set_index('MEAS_DT')
minute = minute_by_lot[LOT]
lot_start = lot_bounds.loc[LOT, 'start']
lot_end = lot_bounds.loc[LOT, 'end']
grade = lot_bounds.loc[LOT, 'grade']
same_grade_lots = lot_bounds.index[lot_bounds.grade.eq(grade)].tolist()

## 이 검토에 실제 사용한 관측 수

| 센서 | 원시 유효 관측 | 원시 결측 | 전체 1분 칸 | 30개 이상인 분 | 30개 미만인 분 |
| --- | --- | --- | --- | --- | --- |
| TC-OUT1 | 38237 | 0 | 652 | 651 | 1 |
| TC-OUT2 | 38236 | 1 | 652 | 651 | 1 |
| TC-Z3 | 38237 | 0 | 652 | 651 | 1 |
| TC-Z8 | 38237 | 0 | 652 | 651 | 1 |
| OP-Z4 | 38237 | 0 | 652 | 651 | 1 |

아래 범위 표는 비교의 배경이다. 센서 단위가 다르므로 숫자의 크기를 센서끼리 직접 비교하지 않는다.

| 센서 | 1분 최솟값 | 1분 중앙값 | 1분 최댓값 | 원시 최솟값 | 원시 최댓값 |
| --- | --- | --- | --- | --- | --- |
| TC-OUT1 | 271.301 | 285.875 | 298.684 | 271.25 | 298.759 |
| TC-OUT2 | 271.005 | 281.489 | 292.359 | 270.952 | 292.441 |
| TC-Z3 | 852.632 | 860.992 | 872.326 | 852.541 | 872.549 |
| TC-Z8 | 333.248 | 333.53 | 333.696 | 333.186 | 333.762 |
| OP-Z4 | 50.31 | 56.837 | 59.858 | 46.2785 | 64.9201 |

In [ ]:
sampling_rows = []

def readable_table(table):
    # CP의 작은 변화는 보존하고 온도·출력 표의 불필요한 소수 자릿수는 줄입니다.
    shown = table.astype(object).copy()
    for row_index, row in table.iterrows():
        sensor = str(row.get('센서', ''))
        for column, value in row.items():
            if isinstance(value, (float, np.floating)) and pd.notna(value):
                digits = 3
                if sensor.startswith('CP-'):
                    digits = 6
                elif sensor.startswith('OP-') and str(column).startswith('원시'):
                    digits = 4
                shown.at[row_index, column] = f'{value:.{digits}f}'.rstrip('0').rstrip('.')
    return shown

focus_rows = []
for sensor in FOCUS:
    source = raw_lot[sensor]
    means = minute[sensor]
    counts = count_by_lot[LOT][sensor]
    sampling_rows.append({
        '센서': sensor,
        '원시 유효 관측': int(source.notna().sum()),
        '원시 결측': int(source.isna().sum()),
        '전체 1분 칸': len(means),
        '30개 이상인 분': int(counts.ge(30).sum()),
        '30개 미만인 분': int(counts.lt(30).sum()),
    })
    focus_rows.append({
        '센서': sensor,
        '1분 최솟값': means.min(),
        '1분 중앙값': means.median(),
        '1분 최댓값': means.max(),
        '원시 최솟값': source.min(),
        '원시 최댓값': source.max(),
    })
sampling_table = pd.DataFrame(sampling_rows)
focus_table = pd.DataFrame(focus_rows)
display(sampling_table)
display(readable_table(focus_table))

## 같은 강종과 실제 정비 기록의 비교 준비

아래 비교값은 관측한 자료를 요약한 수치다. 미래 자료를 포함할 수 있으므로 학습 입력의 기준값으로 그대로 사용하지 않는다.

In [ ]:
# 같은 강종도 LOT별로 보여 줍니다. 긴 LOT에 가중된 강종 평균은 만들지 않습니다.
comparison_rows = []
comparison_values = {}
for lot_no in same_grade_lots:
    data = minute_by_lot[lot_no]
    scope = '관측 전체'
    if grade == 'SPCEN' and lot_no == 240114:
        data = data.loc[data.index < pd.Timestamp('2024-11-04 20:10')]
        scope = '20:10 이전: 후반 하강 전'
    comparison_values[lot_no] = data
    for sensor in COMPARE:
        series = data[sensor].dropna()
        comparison_rows.append({
            'LOT': lot_no,
            '시작일': lot_bounds.loc[lot_no, 'start'].strftime('%Y-%m-%d'),
            '비교 범위': scope,
            '센서': sensor,
            '유효 분': len(series),
            '25%': series.quantile(.25),
            '중앙값': series.median(),
            '75%': series.quantile(.75),
        })
comparison_table = pd.DataFrame(comparison_rows)

In [ ]:
# 같은 설비의 실제 기록만 연결합니다. 가까운 기록도 원인이라는 뜻은 아닙니다.
cal = maintenance.loc[maintenance.EQP_TAG.eq('P1-CR1-CAL01')].copy()
detection_inside = cal.DETC_DT.between(lot_start, lot_end)
work_overlaps = cal.STRT_DT.le(lot_end) & cal.END_DT.ge(lot_start)
related = cal.loc[detection_inside | work_overlaps].copy()
previous = cal.loc[cal.DETC_DT.lt(lot_start)].sort_values('DETC_DT').tail(1)
following = cal.loc[cal.DETC_DT.gt(lot_end)].sort_values('DETC_DT').head(1)
previous_work = cal.loc[cal.END_DT.lt(lot_start)].sort_values('END_DT').tail(1)
following_work = cal.loc[cal.STRT_DT.gt(lot_end)].sort_values('STRT_DT').head(1)
context = pd.concat([previous, previous_work, related, following, following_work])
context = context.drop_duplicates('WO_NO').sort_values('DETC_DT')
maintenance_names = {
    'TBM': '정기점검(TBM)',
    'BM': '고장 후 정비(BM)',
    'CBM': '상태기반 정비(CBM)',
}
change_names = part_changes.merge(
    parts[['PART_CD', 'PART_NM']],
    on='PART_CD',
    how='left',
    validate='many_to_one',
)
context_rows = []
for _, record in context.iterrows():
    if pd.isna(record.DETC_DT):
        relation = '감지 시각 없음'
    elif record.DETC_DT < lot_start:
        relation = f'LOT 시작 {((lot_start-record.DETC_DT).total_seconds()/3600):.2f}시간 전 감지'
    elif record.DETC_DT > lot_end:
        relation = f'LOT 종료 {((record.DETC_DT-lot_end).total_seconds()/3600):.2f}시간 후 감지'
    else:
        relation = '관측 중 감지'
    if record.STRT_DT <= lot_end and record.END_DT >= lot_start:
        relation += '; 작업 시간도 관측과 겹침'
    elif record.END_DT < lot_start:
        relation += f'; 작업 끝은 시작 {(lot_start-record.END_DT).total_seconds()/3600:.2f}시간 전'
    elif record.STRT_DT > lot_end:
        relation += f'; 작업 시작은 종료 {(record.STRT_DT-lot_end).total_seconds()/3600:.2f}시간 후'
    changes = change_names.loc[change_names.WO_NO.eq(record.WO_NO)]
    items = []
    for _, change in changes.iterrows():
        items.append(f'{change.PART_NM} {change.QTY}개({change.CHG_RSN})')
    context_rows.append({
        '작업번호': record.WO_NO,
        '유형': maintenance_names.get(record.MNT_TYPE, record.MNT_TYPE),
        '감지': record.DETC_DT,
        '작업 시작': record.STRT_DT,
        '작업 끝': record.END_DT,
        '관측과의 관계': relation,
        '기록 내용': record.REMARK if pd.notna(record.REMARK) else '기재 없음',
        '교체': ', '.join(items) if items else '교체 행 없음',
    })
context_table = pd.DataFrame(context_rows)
lot_events = events.loc[events.PLANT_CD.eq('CR1') & events.EVT_DT.between(lot_start, lot_end)]
event_table = lot_events[['EVT_ID', 'EVT_DT', 'EVT_TYPE', 'BEF_VAL', 'AFT_VAL']].copy()
cal_repairs = repair.loc[repair.PLANT_CD.eq('CR1')].copy()
# 종료 당일을 포함합니다. 관측 중 겹침과 가까운 계획을 구분합니다.
repair_overlaps = cal_repairs.STRT_DT.le(lot_end) & (cal_repairs.END_DT+pd.Timedelta(days=1)).gt(lot_start)

markers = []
for _, record in related.iterrows():
    if lot_start <= record.DETC_DT <= lot_end:
        label = maintenance_names.get(record.MNT_TYPE, record.MNT_TYPE)
        markers.append((record.DETC_DT, f'{label} 감지', '#a22b45'))

## 질문에 답하기 위한 실제 수치

| 확인 항목 | 재계산 결과 | 해석할 때 주의 |
| --- | --- | --- |
| TC-OUT2 1분 범위 | 271.005~292.359 ℃ | 단조 상승이 아닌 반복 파형 확인 |
| TC-OUT2 중앙값: 121 / 037 / 133 | 281.489 / 281.791 / 279.045 ℃ | 관측 길이가 다른 LOT별 비교 |
| 확대 구간 최고 시각 | TC-Z3 16:42 / TC-OUT1 16:49 / TC-OUT2 16:50 | 단일 구간 시차를 이동 시간으로 단정하지 않음 |
| TC-Z8 중앙값: 121 / 037 / 133 | 333.530 / 330.763 / 330.291 ℃ | 높은 수준만으로 제외하지 않음 |

**관찰:** 냉각 온도는 관측 전체에서 오르내림을 반복한다. 확대 구간에서 가열과 냉각 최고점에 시차가 있지만 TC-Z8은 다른 SPCE보다 높은 수준에 있다.

**가능한 해석과 한계:** 최고점 사이의 시차는 관찰이며 소재 이동 시간이나 인과관계의 검증이 아니다. 과시효 온도의 높은 수준도 검증된 이상 한계를 넘었다는 뜻이 아니다.

In [ ]:
def median(lot_no, channel, start=None, end=None):
    data = minute_by_lot[lot_no]
    if start is not None:
        data = data.loc[data.index >= pd.Timestamp(start)]
    if end is not None:
        data = data.loc[data.index < pd.Timestamp(end)]
    return data[channel].median()

def at(channel, timestamp):
    return minute.loc[pd.Timestamp(timestamp), channel]

def raw_extreme(channel, maximum=True):
    series = raw_lot[channel].dropna()
    timestamp = series.idxmax() if maximum else series.idxmin()
    return timestamp, series.loc[timestamp]

def persistent_start(condition):
    # 유효 분만 넘겨 받아 마지막 유효 분까지 유지된 조건을 찾습니다.
    # 관측 종료 뒤는 알 수 없으며, 실시간 판단 규칙이 아닙니다.
    condition = condition.fillna(False)
    last_run = condition.iloc[::-1].cummin().iloc[::-1]
    times = last_run.index[last_run]
    return times[0] if len(times) else pd.NaT

zoom = minute.loc[ZOOM_START:ZOOM_END]
peak_times = [f'{sensor} {zoom[sensor].idxmax():%H:%M}' for sensor in ['TC-Z3', 'TC-OUT1', 'TC-OUT2']]
evidence_rows = [
    ('TC-OUT2 1분 범위', f"{minute['TC-OUT2'].min():.3f}~{minute['TC-OUT2'].max():.3f} ℃", '단조 상승이 아닌 반복 파형 확인'),
    (
        'TC-OUT2 중앙값: 121 / 037 / 133',
        (
            f"{median(240121, 'TC-OUT2'):.3f} / {median(240037, 'TC-OUT2'):.3f}"
            f" / {median(240133, 'TC-OUT2'):.3f} ℃"
        ),
        '관측 길이가 다른 LOT별 비교',
    ),
    ('확대 구간 최고 시각', ' / '.join(peak_times), '단일 구간 시차를 이동 시간으로 단정하지 않음'),
    (
        'TC-Z8 중앙값: 121 / 037 / 133',
        (
            f"{median(240121, 'TC-Z8'):.3f} / {median(240037, 'TC-Z8'):.3f}"
            f" / {median(240133, 'TC-Z8'):.3f} ℃"
        ),
        '높은 수준만으로 제외하지 않음',
    ),
]

evidence_table = pd.DataFrame(evidence_rows, columns=['확인 항목', '재계산 결과', '해석할 때 주의'])
display(evidence_table)
for title, table in extra_tables:
    print(title)
    display(readable_table(table))

## 다른 센서에서도 확인한 변화

본문에서 다룬 센서 외에도 LOT 끝부분의 건조 센서를 확인했다. 처음과 끝의 중앙값이 비슷하더라도 말기에는 온도가 올랐다 내려가고 출력이 반대로 반응할 수 있어, 끝 15분의 범위도 함께 표시한다. 이를 본문의 CP 급변이나 뒤늦은 정비와 같은 원인으로 묶지 않는다.

| 센서 | 첫 15분 유효 분 | 끝 15분 유효 분 | 첫 15분 중앙값 | 끝 15분 중앙값 | 끝−처음 | 끝 15분 최저 | 끝 15분 최고 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z1 | 15 | 14 | 99.868 | 99.126 | -0.742 | 98.382 | 100.609 |
| TC-Z2 | 15 | 14 | 100.279 | 99.044 | -1.235 | 98.613 | 100.168 |
| OP-Z1 | 15 | 14 | 76.03 | 73.794 | -2.236 | 68.661 | 77.472 |
| OP-Z2 | 15 | 14 | 26.321 | 24.805 | -1.516 | 14.297 | 25.946 |

온도는 ℃, 출력은 % 가정이다. 첫·끝 15분은 시간 칸 기준이며 센서 결측으로 유효 분 수가 다를 수 있다.

In [ ]:
RELATED_SENSORS = ['TC-Z1', 'TC-Z2', 'OP-Z1', 'OP-Z2']

related_rows = []
for sensor in RELATED_SENSORS:
    values = minute[sensor]
    first = values.iloc[:15].dropna()
    last = values.iloc[-15:].dropna()
    related_rows.append({
        '센서': sensor,
        '첫 15분 유효 분': len(first),
        '끝 15분 유효 분': len(last),
        '첫 15분 중앙값': first.median(),
        '끝 15분 중앙값': last.median(),
        '끝−처음': last.median() - first.median(),
        '끝 15분 최저': last.min(),
        '끝 15분 최고': last.max(),
    })
related_table = pd.DataFrame(related_rows)
display(readable_table(related_table))

## 전체 관측에서 이 현상이 차지하는 위치

냉각 온도는 관측 전체에서 오르내림을 반복한다. 확대 구간에서 가열과 냉각 최고점에 시차가 있지만 TC-Z8은 다른 SPCE보다 높은 수준에 있다.

옅은 선은 원시 관측, 진한 선은 유효 1분 평균이다. 노란 영역은 다음 그림에서 확대할 범위이며 학습 제외 표시가 아니다. 다른 단위의 센서는 축을 분리했다.

![전체 관측](../../figures/07_lot_anomaly_review_namelesspark/13_LOT_240121/01_overview.png)

In [ ]:
def sensor_unit(sensor):
    if sensor.startswith('CP-'):
        return '단위 미확인'
    if sensor.startswith('OP-'):
        return '% (가정)'
    return '℃ (가정)'

def draw_timeseries(start, end, filename, title, overview=False):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    fig, axes = plt.subplots(len(FOCUS), 1, figsize=(12, 2.05*len(FOCUS)+1.0), sharex=True)
    axes = np.atleast_1d(axes)
    source = raw_lot.loc[start:end]
    averages = minute.loc[start.floor('min'):end.ceil('min')]
    # 없는 초를 NaN으로 둬 원시 관측 공백도 선으로 이어 붙이지 않습니다.
    grid = pd.date_range(start.ceil('s'), end.floor('s'), freq='1s')
    source = source.reindex(grid)
    for ax, sensor in zip(axes, FOCUS):
        ax.plot(source.index, source[sensor], color='#9bbbd1', lw=.6, alpha=.55, label='원시 관측')
        # 표의 13:29는 13:29:00~59 평균입니다. 곡선은 그 분의 가운데에 표시합니다.
        average_times = averages.index + pd.Timedelta(seconds=30)
        ax.plot(average_times, averages[sensor], color='#153f69', lw=1.2, label='1분 평균 (분 가운데, 관측≥30)')
        if overview:
            ax.axvspan(pd.Timestamp(ZOOM_START), pd.Timestamp(ZOOM_END), color='#e3bd65', alpha=.18, label='아래에서 확대할 구간')
        for timestamp, label, color in markers:
            if '1분 최고' in label:
                timestamp = timestamp + pd.Timedelta(seconds=30)
            if start <= timestamp <= end:
                ax.axvline(timestamp, color=color, lw=.9, ls='--', alpha=.85, label=label)
        for _, record in related.iterrows():
            left = max(start, record.STRT_DT)
            right = min(end, record.END_DT)
            if left < right:
                label = maintenance_names.get(record.MNT_TYPE, record.MNT_TYPE)
                ax.axvspan(left, right, color='#b5a5cc', alpha=.13, label=f'{label} 작업')
        ax.set_ylabel(f'{sensor}\n{sensor_unit(sensor)}', fontsize=10)
        ax.tick_params(labelsize=9)
        ax.grid(alpha=.18, lw=.5)
        ax.set_xlim(start, end)
    handles, labels = axes[0].get_legend_handles_labels()
    unique = {}
    for handle, label in zip(handles, labels):
        if label not in unique:
            unique[label] = handle
    fig.legend(unique.values(), unique.keys(), loc='upper center', bbox_to_anchor=(.5, .955), ncol=3, fontsize=9, frameon=False)
    locator = mdates.AutoDateLocator(minticks=4, maxticks=9)
    axes[-1].xaxis.set_major_locator(locator)
    axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    axes[-1].set_xlabel('관측 시각 — 색칠은 검토 범위이며 자동 학습 제외 표시가 아닙니다.', fontsize=9)
    fig.suptitle(f'LOT {LOT} · {grade} | {PLOT_TITLE}\n{title}', fontsize=13, y=.995)
    fig.subplots_adjust(left=.115, right=.98, bottom=.075, top=.89, hspace=.28)
    figure_path = Path(FIG_DIR) / filename
    fig.savefig(figure_path, dpi=160, facecolor='white')
    plt.show()
    plt.close(fig)
    return figure_path

def draw_comparison():
    if len(same_grade_lots) < 2:
        print(f'{grade}은 관측 LOT가 하나여서 같은 강종의 독립 비교가 불가능합니다.')
        return None
    fig, axes = plt.subplots(len(COMPARE), 1, figsize=(10, 3.0*len(COMPARE)), squeeze=False)
    labels = []
    for lot_no in same_grade_lots:
        label = str(lot_no)
        if grade == 'SPCEN' and lot_no == 240114:
            label += '\n20:10 하강 전'
        labels.append(label)
    for ax, sensor in zip(axes[:, 0], COMPARE):
        values = [comparison_values[lot_no][sensor].dropna().to_numpy() for lot_no in same_grade_lots]
        boxes = ax.boxplot(values, tick_labels=labels, showfliers=False, widths=.45, patch_artist=True)
        for patch, lot_no in zip(boxes['boxes'], same_grade_lots):
            patch.set_facecolor('#82afca' if lot_no == LOT else '#dce6ec')
        ax.set_ylabel(f'{sensor}\n{sensor_unit(sensor)}')
        ax.grid(axis='y', alpha=.2)
        ax.set_title(f'{sensor}: LOT별 분포 (상자 25~75%, 가운데 선 중앙값)', fontsize=10)
    axes[-1, 0].set_xlabel('LOT — 극단점 생략. 원시 최댓값은 표·시계열에서 확인하세요.\n서로 다른 날짜·관측 길이·운전 조건을 포함합니다.')
    fig.suptitle(f'{grade} {len(same_grade_lots)}개 LOT 비교 · 진한 상자 = {LOT}', fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, .95])
    path = Path(FIG_DIR) / '03_same_grade.png'
    fig.savefig(path, dpi=160, facecolor='white')
    plt.show()
    plt.close(fig)
    return path

overview_path = draw_timeseries(lot_start, lot_end, '01_overview.png', '전체 관측과 검토 구간', overview=True)

## 원시값까지 확대해서 확인

확대 범위: **2024-11-24 16:30 ~ 2024-11-24 17:05**. 이 범위에 감지·작업 기록이 있으면 감지 시각은 점선, 작업 시간은 보라색으로 표시한다. 별도로 비교한 시각도 점선으로 표시하며 뜻은 범례에 적었다.

최고점 사이의 시차는 관찰이며 소재 이동 시간이나 인과관계의 검증이 아니다. 과시효 온도의 높은 수준도 검증된 이상 한계를 넘었다는 뜻이 아니다.

![확대 관측](../../figures/07_lot_anomaly_review_namelesspark/13_LOT_240121/02_zoom.png)

In [ ]:
zoom_path = draw_timeseries(ZOOM_START, ZOOM_END, '02_zoom.png', '관심 구간의 원시 관측과 1분 평균')

## 같은 강종 비교

같은 강종 **3개 LOT**: 240037, 240121, 240133. 각 LOT를 따로 비교하며, 긴 LOT에 가중된 강종 전체 평균은 만들지 않았다. 날짜·관측 길이·운전 상태·소재 부하가 다르므로 강종 효과만 분리한 비교는 아니다.

| LOT | 시작일 | 비교 범위 | 센서 | 유효 분 | 25% | 중앙값 | 75% |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 240037 | 2024-04-13 | 관측 전체 | TC-OUT2 | 611 | 275.482 | 281.791 | 287.662 |
| 240037 | 2024-04-13 | 관측 전체 | TC-Z8 | 611 | 330.728 | 330.763 | 330.798 |
| 240121 | 2024-11-24 | 관측 전체 | TC-OUT2 | 651 | 274.977 | 281.489 | 287.966 |
| 240121 | 2024-11-24 | 관측 전체 | TC-Z8 | 651 | 333.494 | 333.53 | 333.569 |
| 240133 | 2024-12-28 | 관측 전체 | TC-OUT2 | 107 | 275.045 | 279.045 | 284.77 |
| 240133 | 2024-12-28 | 관측 전체 | TC-Z8 | 107 | 330.24 | 330.291 | 330.353 |

분포 그림은 극단점을 생략했다. 최대·최소와 짧은 급변은 위 수치표와 시계열에서 확인한다.

![같은 강종 분포](../../figures/07_lot_anomaly_review_namelesspark/13_LOT_240121/03_same_grade.png)

In [ ]:
display(readable_table(comparison_table))
comparison_path = draw_comparison()

## 조업·정비 기록은 무엇을 뒷받침하는가

CAL01의 관측 중 감지·작업 중첩과 가장 가까운 이전/이후 감지·작업 기록을 구분했다. 가까운 기록을 자동 원인으로 연결하지 않는다. 정기점검(TBM), 고장 후 정비(BM), 상태기반 정비(CBM)는 기록의 유형이다. 정비이력의 감지 시각은 실제 이상 시작 시각과 같다고 볼 수 없으며, 특히 정기점검(TBM)의 감지를 고장 발생으로 해석하지 않는다. 정기수리 캘린더는 계획 기간이며, 실제 작업 시각은 별도의 정비이력과 구분한다.

| 작업번호 | 유형 | 감지 | 작업 시작 | 작업 끝 | 관측과의 관계 | 기록 내용 | 교체 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-01854 | 상태기반 정비(CBM) | 2024-11-12 15:00:00 | 2024-11-13 08:54:00 | 2024-11-13 12:44:00 | LOT 시작 280.52시간 전 감지; 작업 끝은 시작 258.78시간 전 | 마모량 초과 | 교체 행 없음 |
| WO-2024-01859 | 정기점검(TBM) | 2024-11-13 07:13:00 | 2024-11-13 07:40:00 | 2024-11-13 09:35:00 | LOT 시작 264.30시간 전 감지; 작업 끝은 시작 261.93시간 전 | 계획 점검 수행 | 교체 행 없음 |
| WO-2024-01922 | 상태기반 정비(CBM) | 2024-11-25 14:19:00 | 2024-11-30 04:19:00 | 2024-11-30 07:12:00 | LOT 종료 19.95시간 후 감지; 작업 시작은 종료 129.95시간 후 | 1호기 재발, 설비 트립 | 교체 행 없음 |

관측 안에 있는 CR1 조업이벤트:

| EVT_ID | EVT_DT | EVT_TYPE | BEF_VAL | AFT_VAL |
| --- | --- | --- | --- | --- |
| EVT-2024-0238 | 2024-11-24 07:31:00 | 강종변경 | SPCD | SPCE |

이 CSV의 CR1 이벤트 유형은 강종변경이다. 이 기록만으로 출력 설정, 재가열·정지 명령, 소재 이동 속도를 확인할 수 없다.

관측과 겹치는 CR1 정기수리 계획은 **0건**이다. 아래 계획의 종료일은 당일까지 포함한다. 계획이 가깝다는 사실과 센서 변화 원인은 구분한다.

| PLAN_ID | STRT_DT | END_DT | SCOPE |
| --- | --- | --- | --- |
| PLAN-2024-02 | 2024-06-10 00:00:00 | 2024-06-14 00:00:00 | 연속소둔로 노내 정비 |
| PLAN-2024-04 | 2024-11-05 00:00:00 | 2024-11-10 00:00:00 | 냉연 라인 전 설비 정기수리 |

In [ ]:
display(context_table)
display(event_table)
print('관측과 겹친 CR1 정기수리 계획:', int(repair_overlaps.sum()), '건')
display(cal_repairs[['PLAN_ID', 'STRT_DT', 'END_DT', 'SCOPE']])

## 학습에 넣을지에 대한 현재 판단

시험 자료를 유지한다. 높은 TC-Z8 또는 반복적인 냉각 온도만으로 제외하지 않는다. 같은 강종 비교에서도 관측 길이와 다른 시기라는 한계를 함께 기록한다.

관찰된 차이, 가능한 원인, 모델 입력·학습 판단은 서로 구분한다. 이 노트북의 색칠과 감지선은 정답 라벨이 아니다. 단일 LOT·강종별 소수 LOT·간헐 관측이라는 한계를 유지한다.

[전체 LOT 해석(노션)](https://app.notion.com/p/3f7e609a45d080b88934e41f4403a466) · [05 학습 제외 기준](../05_exclusion_windows_namelesspark.ipynb)